# Step 7 — Incremental load and correction test

This notebook tests the next-day order feed against both implementations. ETL merges the increment before validation; ELT appends every increment row to raw storage and uses the latest `order_id` version in SQL staging.

Expected behavior: 150 new orders are inserted, 10 existing orders are updated, raw ELT rows increase by 160, and staged order IDs remain unique.

In [ ]:
from pathlib import Path
import sys
import duckdb

# Import both tested implementations so the notebook compares the two designs.
sys.path.insert(0, str(Path('include').resolve()))
from nammamart_etl import run_etl
from nammamart_elt import run_elt, run_incremental_elt

etl_warehouse = Path('assignment/warehouse_samy_etl_step7.duckdb')
elt_warehouse = Path('assignment/warehouse_samy_elt_step7.duckdb')
etl_quarantine = Path('assignment/quarantine_samy_step7')

# First create each September baseline before applying October's increment.
etl_initial = run_etl(warehouse_path=etl_warehouse, quarantine_dir=etl_quarantine, run_id='step7-etl-initial')
elt_initial = run_elt(warehouse_path=elt_warehouse, run_id='step7-elt-initial')
print(f"ETL baseline fact rows: {etl_initial['rows_loaded']:,}")
print(f"ELT baseline staged rows: {elt_initial['stg_orders']:,}")

In [ ]:
# Apply the increment to both designs and print insert/update metrics.
etl_increment = run_etl(warehouse_path=etl_warehouse, quarantine_dir=etl_quarantine, run_id='step7-etl-increment', include_increment=True)
elt_increment = run_incremental_elt(warehouse_path=elt_warehouse, run_id='step7-elt-increment')

print('ETL incremental result')
print(f"  inserted: {etl_increment['increment_inserted']:,}")
print(f"  updated:  {etl_increment['increment_updated']:,}")
print(f"  fact rows after merge: {etl_increment['rows_loaded']:,}")
print('ELT incremental result')
print(f"  raw rows appended: {elt_increment['increment_rows_appended']:,}")
print(f"  inserted: {elt_increment['increment_inserted']:,}")
print(f"  updated:  {elt_increment['increment_updated']:,}")
print(f"  raw rows after append: {elt_increment['raw_orders']:,}")
print(f"  staged unique rows after upsert: {elt_increment['stg_orders']:,}")

In [ ]:
# Prove that ELT raw history keeps all rows while staging exposes one latest row per order.
con = duckdb.connect(str(elt_warehouse), read_only=True)
duplicate_staged_ids = con.execute("""
    SELECT count(*) FROM (
        SELECT order_id FROM stg_orders GROUP BY order_id HAVING count(*) > 1
    )
""").fetchone()[0]
correction_sample = con.execute("""
    SELECT order_id, order_status, _source_file
    FROM raw_orders
    WHERE _source_file = 'orders_2026-10-01_increment.csv'
    ORDER BY order_id
    LIMIT 10
""").fetchdf()
con.close()
print(f'Staged duplicate order IDs: {duplicate_staged_ids}')
display(correction_sample)
print('Raw history is preserved; the SQL latest-row rule prevents duplicate staged orders.')